# 02 Data preparation — reviewed notebook

The earlier cells retain the filename-parsing and inventory exercises. All labelling cells now use the same seven-bearing plan. Intermediate exercises no longer write over the final registry.

**Run order:** restart the project kernel and Run All. Check the Windows project path. The audit section reloads the saved CSV if present, then keeps only the nine base columns. If the CSV is absent, it uses the inventory built above. Derived settings and audits are rebuilt later.

**Limits:** real `.mat` files are not attached here. Review validation uses synthetic MATLAB fixtures, not your real recordings. The reported 79 timing-review flags and unverified amplitude units remain unresolved; no samples are altered or excluded.

**Checks completed here:** notebook schema and code syntax; full synthetic-data execution from raw files; fresh-kernel rerun using an enriched CSV (560 rows, 26 final columns, no `_x`/`_y` audit columns); and rejection of missing audit results before final writeback. Synthetic flags are not the reported real-data counts.

In [ ]:
from pathlib import Path
raw_folder = Path(
    r"C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection"
    r"\data\raw\paderborn"
)
print("folder exists:", raw_folder.exists())

In [ ]:
#find the mat files
files  =sorted(raw_folder.glob("**/*.mat"))
print(len(files), "files found")

In [ ]:
#inspect the first 5 files
for file in files[:5]:
    print(file.name)


In [ ]:
#split the files names from its parts
recording_name = files[0].stem
parts = recording_name.split("_")
parts

In [ ]:
#extract bearning id and recording no
bearing_id  = parts[3]
recording_no = parts[4]

print("bearing id:", bearing_id)
print("recording no:", recording_no)

In [ ]:
#extract the operating condition code 
speed_code = parts[0]
torq_code = parts[1]
force_code = parts[2]
print("speed code:", speed_code)
print("torq code:", torq_code)
print("force code:", force_code)

In [ ]:
record = {
    "bearing_id": bearing_id,
    "recording_no": recording_no,
    "speed_code": speed_code,
    "torq_code": torq_code,
    "force_code": force_code,
    "relative_path": files[0].relative_to(raw_folder).as_posix()
}

In [ ]:
record

In [ ]:
#create registry of all recordings
records = []
for file in files:
    recording_name = file.stem
    parts = recording_name.split("_")
    bearing_id  = parts[3]
    recording_no = parts[4]
    speed_code = parts[0]
    torq_code = parts[1]
    force_code = parts[2]
    
    record = {
        "bearing_id": bearing_id,
        "recording_no": recording_no,
        "speed_code": speed_code,
        "torq_code": torq_code,
        "force_code": force_code,
        "relative_path": file.relative_to(raw_folder).as_posix()
    }
    
    records.append(record)
print("registry entries:", len(records))

In [ ]:
#convert registry to a table
import pandas as pd
registry = pd.DataFrame(records)
print("rows:", registry.shape[0], "columns:", registry.shape[1])
registry.head()

In [ ]:
registry['recording_id']=[Path(path).stem for path in registry["relative_path"]]

In [ ]:
registry.head()

In [ ]:
#check to duplicate recording ids
duplicate_recording_ids = registry['recording_id'].duplicated()
print("duplicate recording ids:", duplicate_recording_ids.sum())


In [ ]:
#count recordings of each bearing
registry['bearing_id'].value_counts()

In [ ]:
#counts recording of each operating condition
condition_counts = registry.groupby(['speed_code', 'torq_code', 'force_code']).size()
print(condition_counts)

In [ ]:
registry = registry.sort_values(by=['bearing_id', 'speed_code', 'torq_code', 'force_code', 'recording_no']).reset_index(drop=True)
registry.head()

In [ ]:
registry["recording_no"] = registry["recording_no"].astype(int)

registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

print(registry[["recording_no", "recording_id"]].head())

In [ ]:
# Use the complete fixed plan, even when rerunning earlier learning cells.
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy",
    "K003": "healthy",
    "KA04": "faulty",
    "KA15": "faulty",
    "KI04": "faulty",
    "KI14": "faulty"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development",
    "KA04": "development",
    "KI04": "development",
    "K003": "final_test",
    "KA15": "final_test",
    "KI14": "final_test"
})

print(registry.groupby(["partition", "condition"]).size())

In [ ]:
output_path = raw_folder.parents[2] / "metadata" / "recording_registry.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
print("Registry destination:", output_path)
print("The complete registry will be saved in the final cell.")

In [ ]:
# Preview the current table; this is not a disk-readback check.
saved_registry = registry.copy()
print("Working rows and columns:", saved_registry.shape)
print(saved_registry.groupby(["bearing_id", "condition", "partition"]).size())

In [ ]:
k002_files = list(raw_folder.rglob("*_K002_*.mat"))

print("K002 recordings found:", len(k002_files))

In [ ]:
files = sorted(raw_folder.rglob("*.mat"))

print("Total recordings:", len(files))

In [ ]:
records = []

for file in files:
    parts = file.stem.split("_")

    records.append({
        "recording_id": file.stem,
        "bearing_id": parts[3],
        "recording_no": int(parts[4]),
        "speed_code": parts[0],
        "torq_code": parts[1],
        "force_code": parts[2],
        "relative_path": file.relative_to(raw_folder).as_posix()
    })

registry = pd.DataFrame(records)

print("Rows and columns:", registry.shape)
print(registry["bearing_id"].value_counts())

In [ ]:
# Use the complete fixed plan, even when rerunning earlier learning cells.
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy",
    "K003": "healthy",
    "KA04": "faulty",
    "KA15": "faulty",
    "KI04": "faulty",
    "KI14": "faulty"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development",
    "KA04": "development",
    "KI04": "development",
    "K003": "final_test",
    "KA15": "final_test",
    "KI14": "final_test"
})

print(registry.groupby(["partition", "condition"]).size())

In [ ]:
print("Duplicate IDs:", registry["recording_id"].duplicated().sum())
print("Missing values:", registry.isna().sum().sum())

In [ ]:
registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

# Save only after the audit joins at the end of the notebook.

print("Working rows:", len(registry))
print("Location:", output_path)

In [ ]:
ka04_files = list(raw_folder.rglob("*_KA04_*.mat"))

print("KA04 recordings found:", len(ka04_files))


In [ ]:
files = sorted(raw_folder.rglob("*.mat"))

print("Total recordings:", len(files))

In [ ]:
records = []

for file in files:
    parts = file.stem.split("_")

    records.append({
        "recording_id": file.stem,
        "bearing_id": parts[3],
        "recording_no": int(parts[4]),
        "speed_code": parts[0],
        "torq_code": parts[1],
        "force_code": parts[2],
        "relative_path": file.relative_to(raw_folder).as_posix()
    })

registry = pd.DataFrame(records)

print(registry["bearing_id"].value_counts())

In [ ]:
# Use the complete fixed plan, even when rerunning earlier learning cells.
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy",
    "K003": "healthy",
    "KA04": "faulty",
    "KA15": "faulty",
    "KI04": "faulty",
    "KI14": "faulty"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development",
    "KA04": "development",
    "KI04": "development",
    "K003": "final_test",
    "KA15": "final_test",
    "KI14": "final_test"
})

print(registry.groupby(["partition", "condition"]).size())

In [ ]:
print("Rows and columns:", registry.shape)
print("Duplicate IDs:", registry["recording_id"].duplicated().sum())
print("Missing values:", registry.isna().sum().sum())

In [ ]:
registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

# Save only after the audit joins at the end of the notebook.

print("Working rows:", len(registry))

In [ ]:
ki04_files = list(raw_folder.rglob("*_KI04_*.mat"))

print("KI04 recordings found:", len(ki04_files))

In [ ]:
files = sorted(raw_folder.rglob("*.mat"))
records = []

for file in files:
    parts = file.stem.split("_")

    records.append({
        "recording_id": file.stem,
        "bearing_id": parts[3],
        "recording_no": int(parts[4]),
        "speed_code": parts[0],
        "torq_code": parts[1],
        "force_code": parts[2],
        "relative_path": file.relative_to(raw_folder).as_posix()
    })

registry = pd.DataFrame(records)

print("Total recordings:", len(registry))
print(registry["bearing_id"].value_counts())

In [ ]:
# Use the complete fixed plan, even when rerunning earlier learning cells.
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy",
    "K003": "healthy",
    "KA04": "faulty",
    "KA15": "faulty",
    "KI04": "faulty",
    "KI14": "faulty"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development",
    "KA04": "development",
    "KI04": "development",
    "K003": "final_test",
    "KA15": "final_test",
    "KI14": "final_test"
})

print(registry.groupby(["partition", "condition"]).size())

In [ ]:
print("Rows and columns:", registry.shape)
print("Duplicate IDs:", registry["recording_id"].duplicated().sum())
print("Missing values:", registry.isna().sum().sum())

In [ ]:
registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

# Save only after the audit joins at the end of the notebook.

print("Working rows:", len(registry))

In [ ]:
k003_files = list(raw_folder.rglob("*_K003_*.mat"))

print("K003 recordings found:", len(k003_files))

In [ ]:
ka15_files = list(raw_folder.rglob("*_KA15_*.mat"))

print("KA15 recordings found:", len(ka15_files))

In [ ]:
ki14_files = list(raw_folder.rglob("*_KI14_*.mat"))

print("KI14 recordings found:", len(ki14_files))

In [ ]:
files = sorted(raw_folder.rglob("*.mat"))
records = []

for file in files:
    parts = file.stem.split("_")

    records.append({
        "recording_id": file.stem,
        "bearing_id": parts[3],
        "recording_no": int(parts[4]),
        "speed_code": parts[0],
        "torq_code": parts[1],
        "force_code": parts[2],
        "relative_path": file.relative_to(raw_folder).as_posix()
    })

registry = pd.DataFrame(records)

print("Total recordings:", len(registry))
print(registry["bearing_id"].value_counts().sort_index())

In [ ]:
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy",
    "K003": "healthy",
    "KA04": "faulty",
    "KA15": "faulty",
    "KI04": "faulty",
    "KI14": "faulty"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development",
    "KA04": "development",
    "KI04": "development",
    "K003": "final_test",
    "KA15": "final_test",
    "KI14": "final_test"
})

print(registry.groupby(["partition", "condition"]).size())

In [ ]:
print("Missing values:", registry.isna().sum().sum())
print("Duplicate IDs:", registry["recording_id"].duplicated().sum())

partitions_per_bearing = registry.groupby("bearing_id")["partition"].nunique()
print("Partitions per bearing:")
print(partitions_per_bearing)

In [ ]:
registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

# Save only after the audit joins at the end of the notebook.

print("Working rows:", len(registry))

In [ ]:
registry.shape

## Reload a clean base table before auditing

**What:** the next cell loads the saved registry when present and selects its nine base columns.

**Why:** the saved CSV may already contain timing and quality columns. Loading those columns and merging fresh audits would produce `_x` and `_y` suffixes. Selecting the base fields prevents that.

**How:** `registry[[...]].copy()` creates an independent table with only the named fields. This does not remove columns from the CSV on disk. The completed `checked_registry` is written only in the last save cell. The file-existence and metadata checks stop the run on an invalid base table.

Expected base size for this project: **560 rows, nine columns**. Final-test files are read only for structural/timing checks.

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path(
    r"C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection"
)
raw_folder = project_root / "data" / "raw" / "paderborn"
registry_path = project_root / "metadata" / "recording_registry.csv"

# Reload a saved registry when available; otherwise use the inventory above.
if registry_path.is_file():
    registry = pd.read_csv(registry_path)

# Start with base columns so later merges do not duplicate audit columns.
registry = registry[[
    "recording_id",
    "bearing_id",
    "recording_no",
    "speed_code",
    "torq_code",
    "force_code",
    "relative_path",
    "condition",
    "partition"
]].copy()

if registry["recording_id"].duplicated().any() or registry.isna().any().any():
    raise ValueError("Check duplicate IDs and missing base metadata.")
if not registry.groupby("bearing_id")["partition"].nunique().eq(1).all():
    raise ValueError("A bearing crosses partition boundaries.")

print("Base rows and columns:", registry.shape)
file_exists = [
    (raw_folder / path).is_file()
    for path in registry["relative_path"]
]
print("Files found:", sum(file_exists))
print("Files missing:", len(file_exists) - sum(file_exists))
if not all(file_exists):
    raise FileNotFoundError("A recording listed in the registry is missing.")

In [ ]:
from scipy.io import loadmat
row = registry[registry["partition"] == "development"].iloc[0]
file_path = raw_folder / row["relative_path"]

recording = loadmat(file_path)[file_path.stem][0, 0]
channels = recording["Y"].ravel()
names = [channel["Name"].item() for channel in channels]

print("Recording:", row["recording_id"])
print("Channels:", names)

In [ ]:
vibration = channels[names.index("vibration_1")]
x = vibration["Data"].ravel()

axis_index = int(vibration["XIndex"].item()) - 1
axis = recording["X"].ravel()[axis_index]
t_stored = axis["Data"].ravel()

print("Vibration samples:", x.size)
print("Time values:", t_stored.size)
print("Channel raster:", vibration["Raster"].item())
print("Axis raster:", axis["Raster"].item())
print("Stored unit:", vibration["Unit"])

In [ ]:
time_span = t_stored[-1] - t_stored[0]
fs_check = (x.size - 1) / time_span

print(f"Stored time span: {time_span:.6f} seconds")
print(f"Rate from time span: {fs_check:.4f} Hz")

In [ ]:
import numpy as np

dt = np.diff(t_stored)

print("Time strictly increasing:", (dt > 0).all())
print("Mean interval (microseconds):", dt.mean() * 1e6)
print("Median interval (microseconds):", np.median(dt) * 1e6)
print("Smallest interval (microseconds):", dt.min() * 1e6)
print("Largest interval (microseconds):", dt.max() * 1e6)

In [ ]:
t_nominal = np.arange(x.size) / 64_000
timing_error = (t_stored - t_stored[0]) - t_nominal

print("Maximum deviation (ms):", np.abs(timing_error).max() * 1000)
print("Final deviation (ms):", timing_error[-1] * 1000)

In [ ]:
comparison_path = (
    raw_folder / "K001" / "K001" / "N15_M07_F10_K001_1.mat"
)

comparison = loadmat(comparison_path)[comparison_path.stem][0, 0]
comparison_channels = comparison["Y"].ravel()

comparison_names = [
    channel["Name"].item() for channel in comparison_channels
]
comparison_vibration = comparison_channels[
    comparison_names.index("vibration_1")
]

comparison_x = comparison_vibration["Data"].ravel()
axis_index = int(comparison_vibration["XIndex"].item()) - 1
comparison_time = comparison["X"].ravel()[axis_index]["Data"].ravel()

span = comparison_time[-1] - comparison_time[0]

print("Comparison samples:", comparison_x.size)
print("Comparison span (s):", span)
print("Comparison axis-derived rate (Hz):", (comparison_x.size - 1) / span)

In [ ]:
def inspect_timing(path):
    recording = loadmat(path)[path.stem][0, 0]
    channels = recording["Y"].ravel()
    names = [channel["Name"].item() for channel in channels]

    vibration = channels[names.index("vibration_1")]
    samples = vibration["Data"].size

    axis_index = int(vibration["XIndex"].item()) - 1
    time = recording["X"].ravel()[axis_index]["Data"].ravel()
    span = time[-1] - time[0]

    return {
        "recording_id": path.stem,
        "sample_count": samples,
        "time_count": time.size,
        "axis_span_s": span,
        "axis_rate_hz": (time.size - 1) / span
    }

In [ ]:
timing_result = inspect_timing(comparison_path)

print(timing_result)

In [ ]:
timing_records = []

for relative_path in registry["relative_path"]:
    path = raw_folder / relative_path
    timing_records.append(inspect_timing(path))

timing_table = pd.DataFrame(timing_records)

print("Recordings inspected:", len(timing_table))
timing_table.head()

In [ ]:
timing_table["axis_rate_difference_pct"] = (
    (timing_table["axis_rate_hz"] - 64_000) / 64_000
).abs() * 100

timing_table.sort_values(
    "axis_rate_difference_pct", ascending=False
).head(10)

In [ ]:
review_threshold_pct = 0.1

timing_table["needs_timing_review"] = (
    timing_table["axis_rate_difference_pct"] > review_threshold_pct
)

print("Recordings checked:", len(timing_table))
print("Flagged for timing review:", timing_table["needs_timing_review"].sum())

In [ ]:
review_id = "N15_M01_F10_K002_10"

review_path = raw_folder / registry.loc[
    registry["recording_id"] == review_id, "relative_path"
].iloc[0]

review = loadmat(review_path)[review_path.stem][0, 0]
review_channels = review["Y"].ravel()
review_names = [channel["Name"].item() for channel in review_channels]
review_vibration = review_channels[review_names.index("vibration_1")]

axis_index = int(review_vibration["XIndex"].item()) - 1
review_axis = review["X"].ravel()[axis_index]
review_time = review_axis["Data"].ravel()
dt = np.diff(review_time)

print("Channel raster:", review_vibration["Raster"].item())
print("Axis raster:", review_axis["Raster"].item())
print("DownSampling:", review_vibration["DownSampling"].item())
print("Time values finite:", np.isfinite(review_time).all())
print("Time strictly increasing:", (dt > 0).all())
print("Interval percentiles (µs):",
      np.percentile(dt, [0, 25, 50, 75, 100]) * 1e6)

In [ ]:
timing_table["timing_status"] = np.where(
    timing_table["needs_timing_review"],
    "needs_review",
    "within_span_tolerance"
)

audit_path = project_root / "metadata" / "timing_audit.csv"
timing_table.to_csv(audit_path, index=False)

print(timing_table["timing_status"].value_counts())

In [ ]:
length_mismatch = (
    timing_table["sample_count"] != timing_table["time_count"]
)

print("Recordings with mismatched lengths:", length_mismatch.sum())

In [ ]:
def inspect_quality(path):
    recording = loadmat(path)[path.stem][0, 0]
    channels = recording["Y"].ravel()
    names = [channel["Name"].item() for channel in channels]

    vibration = channels[names.index("vibration_1")]
    x = vibration["Data"].ravel()

    axis_index = int(vibration["XIndex"].item()) - 1
    axis = recording["X"].ravel()[axis_index]
    time = axis["Data"].ravel()

    return {
        "recording_id": path.stem,
        "signal_finite": np.isfinite(x).all(),
        "time_finite": np.isfinite(time).all(),
        "time_increasing": (np.diff(time) > 0).all(),
        "rasters_match": vibration["Raster"].item() == axis["Raster"].item()
    }

In [ ]:
quality_result = inspect_quality(review_path)

print(quality_result)

In [ ]:
quality_records = []

for relative_path in registry["relative_path"]:
    path = raw_folder / relative_path
    quality_records.append(inspect_quality(path))

quality_table = pd.DataFrame(quality_records)

print("Recordings checked:", len(quality_table))
print(quality_table.drop(columns="recording_id").sum())

In [ ]:
quality_path = project_root / "metadata" / "quality_audit.csv"

quality_table.to_csv(quality_path, index=False)

print("Saved:", quality_path)

In [ ]:
registry["speed_setting_rpm"] = registry["speed_code"].map({
    "N09": 900, "N15": 1500
})

registry["torque_setting_nm"] = registry["torq_code"].map({
    "M01": 0.1, "M07": 0.7
})

registry["radial_force_setting_n"] = registry["force_code"].map({
    "F04": 400, "F10": 1000
})

registry[[
    "speed_setting_rpm",
    "torque_setting_nm",
    "radial_force_setting_n"
]].drop_duplicates()

In [ ]:
registry["vibration_channel"] = "vibration_1"
registry["nominal_fs_hz"] = 64_000
registry["vibration_unit_status"] = "unverified"

registry[[
    "vibration_channel",
    "nominal_fs_hz",
    "vibration_unit_status"
]].drop_duplicates()

In [ ]:
checked_registry = registry.merge(
    timing_table,
    on="recording_id",
    how="left",
    validate="one_to_one"
)

checked_registry = checked_registry.merge(
    quality_table,
    on="recording_id",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(checked_registry))
print(checked_registry["timing_status"].value_counts())

In [ ]:
print("Rows:", len(checked_registry))
print("Missing values:", checked_registry.isna().sum().sum())
print("Duplicate IDs:", checked_registry["recording_id"].duplicated().sum())

In [ ]:
# Stop before writing an incomplete or duplicated registry.
if len(checked_registry) != len(registry):
    raise ValueError("The audit joins changed the recording count.")
if checked_registry.isna().any().any():
    raise ValueError("Missing metadata or audit results: registry was not saved.")
if checked_registry["recording_id"].duplicated().any():
    raise ValueError("Duplicate IDs: registry was not saved.")

registry_path = project_root / "metadata" / "recording_registry.csv"
registry_path.parent.mkdir(parents=True, exist_ok=True)
checked_registry.to_csv(registry_path, index=False)

print("Saved recordings:", len(checked_registry))
print("Location:", registry_path)